# MovieLens EDA for Context-Aware Recommender

This notebook prepares exploratory analysis plots and session synthesis previews for MovieLens 1M.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path('data/raw/movielens/ml-1m/ratings.dat')
OUTPUT_DIR = Path('outputs/figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load ratings.dat (MovieLens 1M)
df = pd.read_csv(
    DATA_PATH,
    sep='::',
    engine='python',
    header=None,
    names=['user_id', 'item_id', 'rating', 'timestamp'],
    encoding='latin-1',
)
df.head()

In [ ]:
# Basic stats
n_users = df['user_id'].nunique()
n_items = df['item_id'].nunique()
n_ratings = len(df)
sparsity = 1 - (n_ratings / (n_users * n_items))

print(f'n_users: {n_users}')
print(f'n_items: {n_items}')
print(f'n_ratings: {n_ratings}')
print(f'sparsity: {sparsity:.6f}')

In [ ]:
# Rating distribution
plt.figure(figsize=(8, 5))
plt.hist(df['rating'], bins=9, edgecolor='black')
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rating_dist.png', dpi=150)
plt.show()

In [ ]:
# User activity (interactions per user, log-scale sorted descending)
user_activity = df.groupby('user_id').size().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
plt.plot(np.arange(len(user_activity)), user_activity.values)
plt.yscale('log')
plt.title('User Activity Long Tail (Log Scale)')
plt.xlabel('Users sorted by activity')
plt.ylabel('Interactions (log scale)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'user_activity.png', dpi=150)
plt.show()

In [ ]:
# Item popularity long-tail plot
item_popularity = df.groupby('item_id').size().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
plt.plot(np.arange(len(item_popularity)), item_popularity.values)
plt.yscale('log')
plt.title('Item Popularity Long Tail (Log Scale)')
plt.xlabel('Items sorted by popularity')
plt.ylabel('Interactions (log scale)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'item_popularity.png', dpi=150)
plt.show()

In [ ]:
# Temporal density by month
df_time = df.copy()
df_time['datetime'] = pd.to_datetime(df_time['timestamp'], unit='s')
df_time['month'] = df_time['datetime'].dt.to_period('M').astype(str)
temporal_density = df_time.groupby('month').size()

plt.figure(figsize=(12, 5))
plt.plot(temporal_density.index, temporal_density.values)
plt.xticks(rotation=90)
plt.title('Interaction Density by Month')
plt.xlabel('Month')
plt.ylabel('Interactions')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'temporal_density.png', dpi=150)
plt.show()

In [ ]:
# Session synthesis preview (gap=3600s)
def synthesize_sessions(frame: pd.DataFrame, gap_seconds: int = 3600) -> pd.DataFrame:
    data = frame.sort_values(['user_id', 'timestamp']).copy()
    data['time_delta'] = data.groupby('user_id')['timestamp'].diff().fillna(gap_seconds + 1)
    data['new_session'] = (data['time_delta'] > gap_seconds).astype(int)
    data['session_id'] = data.groupby('user_id')['new_session'].cumsum()
    data['session_key'] = data['user_id'].astype(str) + '_' + data['session_id'].astype(str)
    data['session_pos'] = data.groupby('session_key').cumcount()
    data['session_len'] = data.groupby('session_key')['session_pos'].transform('count')
    return data

df_session = synthesize_sessions(df, gap_seconds=3600)
session_lengths = df_session[['session_key', 'session_len']].drop_duplicates()['session_len']

print(f'mean session length: {session_lengths.mean():.3f}')
print(f'median session length: {session_lengths.median():.3f}')
print(f'max session length: {session_lengths.max()}')

plt.figure(figsize=(8, 5))
plt.hist(session_lengths, bins=50, edgecolor='black')
plt.title('Session Length Distribution')
plt.xlabel('Session length')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'session_length_dist.png', dpi=150)
plt.show()

In [ ]:
# Timeline sample for 5 random users to validate session synthesis
rng = np.random.default_rng(seed=42)
sample_users = rng.choice(df_session['user_id'].unique(), size=5, replace=False)

fig, axes = plt.subplots(5, 1, figsize=(12, 12), sharex=True)
for idx, user_id in enumerate(sample_users):
    user_df = df_session[df_session['user_id'] == user_id].sort_values('timestamp')
    user_times = pd.to_datetime(user_df['timestamp'], unit='s')
    axes[idx].plot(user_times, user_df['rating'], marker='o', linestyle='-')
    boundaries = user_df[user_df['new_session'] == 1]['timestamp']
    for boundary in boundaries:
        axes[idx].axvline(pd.to_datetime(boundary, unit='s'), color='red', alpha=0.25)
    axes[idx].set_title(f'User {user_id} Timeline')
    axes[idx].set_ylabel('Rating')

axes[-1].set_xlabel('Timestamp')
plt.tight_layout()
plt.show()